In [ ]:
from astropy.io import fits
import matplotlib.pyplot as plt
from astropy.wcs import WCS
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.mast import Observations, Mast
from pprint import pprint
import os

mast = Mast()

target_name = "M42"
search_radius = "0.1 deg"

obs = Observations.query_object(target_name, radius=search_radius)

print(f"MAST rows within {search_radius} of {target_name!r}: {len(obs)}")

if len(obs) == 0:
    raise RuntimeError(
        "No observations returned. Check the target name, your network connection, "
        "and try a larger search_radius (e.g. '0.5 deg')."
    )

miri_obs = obs[
    (obs["instrument_name"] == "MIRI/IMAGE")
    & (obs["dataproduct_type"] == "image")
    & (obs["obs_collection"] == "JWST")
    & (obs["dataRights"] == "PUBLIC")
]

print(
    "After JWST / MIRI/IMAGE / image / PUBLIC filter:",
    len(miri_obs),
)

if len(miri_obs) == 0:
    jwst = obs[obs["obs_collection"] == "JWST"]
    print(f"JWST rows in this cone (any instrument/products): {len(jwst)}")
    if len(jwst) > 0:
        inst = sorted({str(x) for x in jwst["instrument_name"].tolist()})
        preview = inst[:15]
        suffix = " ..." if len(inst) > len(preview) else ""
        print("instrument_name values on those JWST rows:", preview, suffix)
    raise RuntimeError(
        "No JWST MIRI imaging (PUBLIC) matched your filters. "
        "Try another target (e.g. M42), increase search_radius, or relax filters in the next cell."
    )


In [ ]:
miri_obs

In [ ]:
# Filter for MIRI/IFU public observations

# Define the three filters of interest
target_filters = ['F770W', 'F1500W', 'F2550W']

# Create a dictionary to hold results per filter
filtered_by_filter = {}

for f in target_filters:
    filtered_by_filter[f] = miri_obs[miri_obs['filters'] == f]

# Print summary
for f, data in filtered_by_filter.items():
    print(f"\n=== Filter: {f} ===")
    print(f"Count: {len(data)}")
    for row in data:
        print(f"  obs_id: {row['obs_id']} | filters: {row['filters']} | t_exptime: {row['t_exptime']}")

# Optional: combine all three filters into one table
from astropy.table import vstack

_subtables = [filtered_by_filter[f] for f in target_filters if len(filtered_by_filter[f]) > 0]
if len(_subtables) == 0:
    print(
        f"\nNo rows in any of the requested filters {target_filters}. "
        "Edit target_filters or use the full miri_obs table."
    )
    all_filtered = miri_obs[0:0]
else:
    all_filtered = vstack(_subtables)
print(f"\nTotal across requested filters: {len(all_filtered)}")


In [ ]:
for f in target_filters:
    filter_obs = all_filtered[all_filtered['filters'] == f]
    
    if len(filter_obs) == 0:
        print(f"No observations found for filter {f}, skipping...")
        continue
    
    first = filter_obs[0]
    dataURL = first['dataURL']
    
    if not dataURL:
        print(f"No dataURL for filter {f}, skipping...")
        continue

    save_dir = f"./fits_images/{target_name}"
    os.makedirs(save_dir, exist_ok=True)

    filename = dataURL.split("/")[-1]
    local_path = f"{save_dir}/{filename}"

    # Skip if file already exists
    if os.path.exists(local_path):
        print(f"  Already exists, skipping: {local_path}")
        continue

    print(f"Downloading {f}: {filename}")
    Observations.download_file(dataURL, local_path=local_path)
    print(f"  Saved to {local_path}")

In [ ]:
if len(all_filtered) == 0:
    print("Skipping plot: all_filtered is empty (no rows in your chosen filters).")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for i, f in enumerate(target_filters):
        filter_obs = all_filtered[all_filtered['filters'] == f]

        if len(filter_obs) == 0:
            print(f"No observations found for filter {f}, skipping...")
            continue

        dataURL = filter_obs[0]['dataURL']
        filename = dataURL.split("/")[-1]
        local_path = f"./fits_images/{target_name}/{filename}"

        if not os.path.exists(local_path):
            print(f"File not found for filter {f}: {local_path}")
            continue

        hdul = fits.open(local_path)

    # Try named extension first, fall back to index 1
        try:
            cube = hdul['SCI'].data
        except KeyError:
            cube = hdul[1].data

        cube = np.squeeze(cube)  # remove any size-1 dimensions

        if cube.ndim == 3:
            image = np.nansum(cube, axis=0)
        elif cube.ndim == 2:
            image = cube
        else:
            print(f"Cannot handle shape {cube.shape} for filter {f}, skipping...")
            hdul.close()
            continue

        # Replace non-positive values so LogNorm doesn't break
        image = np.where(image <= 0, np.nan, image)

        vmin = np.nanpercentile(image, 5)
        vmax = np.nanpercentile(image, 99)

        # Ensure vmin is positive for LogNorm
        vmin = max(vmin, 1e-10)
        if vmin >= vmax:
            vmax = vmin * 10  # fallback if image is nearly uniform

        ax = axes[i]
        im = ax.imshow(
            image,
            origin='lower',
            cmap='inferno',
            norm=plt.matplotlib.colors.LogNorm(vmin=vmin, vmax=vmax)
        )
        ax.set_title(f"{target_name} — {f}", fontsize=12)
        ax.set_xlabel("X (pixels)")
        ax.set_ylabel("Y (pixels)")
        plt.colorbar(im, ax=ax, label="Flux")
        hdul.close()

    plt.suptitle(f"{target_name} — MIRI IFU Filters", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
